# Зависимости и импорты

In [ ]:
!pip install -U langchain langchain-community chromadb transformers rouge-score datasets faiss-cpu sentence-transformers langchain_huggingface

In [114]:
import os
import json
import random

import pandas as pd # Для сохранения результатов оценки

from datasets import load_dataset # Для загрузки датасетов

from langchain.vectorstores import Chroma # Для работы с ChromaDB
from langchain.embeddings import HuggingFaceEmbeddings # Для эмбеддингов
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig # Для работы с LLM и ее настройкой
from langchain_huggingface import HuggingFacePipeline # Обертка LangChain для HuggingFace pipeline

from rouge_score import rouge_scorer # Для расчета метрики ROUGE-L

import torch # Общий импорт, часто нужен для работы с моделями на GPU

# Инициализация БД и retriever

In [ ]:
dataset_test = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split='test')

In [3]:
# Загружаем весь train-сплит
dataset = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split='passages')


# Преобразуем в список документов
documents = [
    Document(page_content=entry["passage"], metadata={"id": entry["id"]})
    for entry in dataset
]

# Разделим тексты на чанки
splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=50)
chunks = splitter.split_documents(documents)

# Создание эмбеддингов
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

<ipython-input-3-3107530493>:16: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [4]:
db = Chroma.from_documents(chunks, embedding_model, persist_directory="./chroma_db")
db.persist()
retriever = db.as_retriever()

<ipython-input-4-1940448494>:2: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer supported as docs are automatically persisted.
  db.persist()


# Инициализация модели

In [ ]:
model_name = "microsoft/phi-3-mini-4k-instruct"

# Загружаем токенайзер и модель
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
    low_cpu_mem_usage=True
)


In [92]:
# Создаем pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

generation_args = {
    "max_new_tokens": 50,
    "return_full_text": False,
    "temperature": 0.01,
    "do_sample": True,
}

# Оборачиваем в LangChain
llm = HuggingFacePipeline(pipeline=pipe)

print("✅ Модель загружена!")

Device set to use cuda:0


✅ Модель загружена!


# Создание пайплайна ответа и персистентное хранение чата

In [110]:
CHAT_HISTORY_FILE = "chat_history.json"

def load_chat_history():
    """Загружает историю чата из файла."""
    if os.path.exists(CHAT_HISTORY_FILE):
        with open(CHAT_HISTORY_FILE, 'r', encoding='utf-8') as f:
            raw_history = json.load(f)
            return raw_history
    return []

def save_chat_history(history):
    """Сохраняет историю чата в файл."""
    # История уже в нужном формате [{"role": ..., "content": ...}], поэтому просто сохраняем
    with open(CHAT_HISTORY_FILE, 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=4)

def ask_chat(query):
    retrieved_docs = retriever.get_relevant_documents(query)
    context_content = "\n\n".join([doc.page_content for doc in retrieved_docs])
    chat_history = load_chat_history()
    messages = [
    {"role": "system", "content":
     "You are a helpful AI assistant"
     "Бери информацию только из context."
     "Если ее там нет, пиши сразу ЧТО НЕ ЗНАЕШЬ."
     "Отвечай только на заданный вопрос, минимум слов, если можешь ответить в 1-2 - отвечай"
     },
    {"role": "сontext", "content": context_content},
    *chat_history,
    {"role": "user", "content": query},
]

    output = pipe(messages, **generation_args)
    print(output[0]['generated_text'])

    # 5. Добавляем текущий вопрос и ответ модели в историю чата
    chat_history.append({"role": "user", "content": query})
    chat_history.append({"role": "assistant", "content": output[0]['generated_text']})

    # 6. Сохраняем обновленную историю
    save_chat_history(chat_history)

    return output[0]['generated_text']


In [112]:
if os.path.exists(CHAT_HISTORY_FILE):
    os.remove(CHAT_HISTORY_FILE)
    print(f"Очищен файл истории: {CHAT_HISTORY_FILE}")


ask_chat("Привет!")

ask_chat("Кто такой Альберт Эйнштейн?")

ask_chat("О ком я спросил в предыдущем сообщении?")

ask_chat("Сколько лет Земле?")

print('пробные вопросы завершены')

Очищен файл истории: chat_history.json
 Привет!
 Альберт Эйнштейн был одним из самых значимых физиков и cosmologстов в истории. Он родился в 1918 году в Райне, Германии, и умер в 200
 Вы спросили о Альберте Эйнштейне.
 Земля, на которую мы живем, имеет около 4,5 миллиардов лет. Это время, которое она прошла с момента её создания, основанное на основных научных исследованиях
пробные вопросы завершены


# Тестирование на 20 примерах. Метрика - ROUGE-L F-score

In [107]:
if os.path.exists(CHAT_HISTORY_FILE):
    os.remove(CHAT_HISTORY_FILE)
    print(f"Очищен файл истории: {CHAT_HISTORY_FILE} перед началом оценки.")


sampled_indices = random.sample(range(len(dataset_test)), 20)
evaluation_examples = [dataset_test[i] for i in sampled_indices]
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
results = []

for i, example in enumerate(evaluation_examples):
    question = example['question']
    ground_truth_answer = example['answer'] # Эталонный ответ из датасета

    print(f"\n--- Оценка примера {i+1}/{len(evaluation_examples)} ---")
    print(f"Вопрос: {question}")
    print(f"Эталонный ответ: {ground_truth_answer}")

    # Получаем ответ от вашей модели
    model_answer = ask_chat(question)
    print(f"Ответ модели: {model_answer}")

    os.remove(CHAT_HISTORY_FILE)

    # Считаем метрику ROUGE-L
    # Убедимся, что оба аргумента являются строками для rouge_scorer
    scores = scorer.score(str(ground_truth_answer), str(model_answer))
    rouge_l_fscore = scores['rougeL'].fmeasure
    print(f"ROUGE-L F-score: {rouge_l_fscore:.4f}")

    results.append({
        'question': question,
        'ground_truth_answer': ground_truth_answer,
        'model_answer': model_answer,
        'rouge_l_fscore': rouge_l_fscore
    })

# Сохраняем результаты в DataFrame и затем в CSV
results_df = pd.DataFrame(results)
output_csv_path = "evaluation_results.csv"
results_df.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"\nОценка завершена. Результаты сохранены в '{output_csv_path}'")
print("\nПервые 5 строк результатов:")
print(results_df.head())

Очищен файл истории: chat_history.json перед началом оценки.

--- Оценка примера 1/20 ---
Вопрос: What field did Woodrow Wilson leave law practice to study?
Эталонный ответ: history and political science
 Political science
Ответ модели:  Political science
ROUGE-L F-score: 0.6667

--- Оценка примера 2/20 ---
Вопрос: What is surplus killing?
Эталонный ответ: Surplus killing is defined as the killing of several prey animals too numerous to eat at one sitting.
 Surplus killing is a behavior observed in some predators where they kill more prey than they can immediately consume. This can occur for various reasons, such as ensuring a food supply during times when prey is scarce, or when the pred
Ответ модели:  Surplus killing is a behavior observed in some predators where they kill more prey than they can immediately consume. This can occur for various reasons, such as ensuring a food supply during times when prey is scarce, or when the pred
ROUGE-L F-score: 0.1695

--- Оценка примера 3/20 --